In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Importing Required Libraries
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import glob
print("Libraries imported successfully")

In [ ]:
# Loading the Sample image fromt the dataset
image_paths = glob.glob('/kaggle/input/**/*.jpg', recursive=True) + glob.glob('/kaggle/input/**/*.png', recursive=True)
if image_paths:
    sample_image = image_paths[2]
    # print(f"loading image from {image_paths}")

    img = Image.open(sample_image)
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title("Sample image for skin lesion")
    plt.axis("off")
    plt.show()
else:
    print("No image found")

In [ ]:
# List and pair the image and mask path files
train_image_dir = "/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1-2_Training_Input"
mask_dir = "/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1_Training_GroundTruth"

image_paths = sorted(glob.glob(os.path.join(train_image_dir,"*.jpg")))
mask_paths = sorted(glob.glob(os.path.join(mask_dir,"*.png")))

print(f"Images: {len(image_paths)}, Masks: {len(mask_paths)}")
print(image_paths[0])
print(mask_paths[0])

In [ ]:
# Load and preprocess one pair (sanity check)
import cv2

IMG_SIZE = 256

def load_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img / 255.0

def load_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = (mask > 127).astype("float32")
    return mask

img = load_image(image_paths[4])
mask = load_mask(mask_paths[4])

fig, ax = plt.subplots(1,2,figsize=(10,5))
ax[0].imshow(img)
ax[0].set_title("Image")
ax[1].imshow(mask, cmap='gray')
ax[1].set_title("Ground truth")
plt.show()

In [ ]:
# load all the images and masks into the memory
from tqdm import tqdm
X = np.array([load_image(p) for p in tqdm(image_paths)], dtype='float32')
y = np.array([load_mask(p) for p in tqdm(mask_paths)], dtype='float32')
y = np.expand_dims(y,axis=-1)

print("X shape: ", X.shape)
print("Y shape: ", y.shape)

In [ ]:
# split the data into the train test
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape: ", X_train.shape)
print("Test Shape: ", X_val.shape)

In [ ]:
# Importing required libraries for the UNET Architecture
import tensorflow
from tensorflow.keras import layers,Model
print("Libraries imported Successfully")

In [ ]:
# Attention gate function for the Attention UNET Architecture
def attention_gates(x, g, filters):
    
    theta_x = layers.Conv2D(filters, 1, strides=1, padding="same")(x)
    phi_g = layers.Conv2D(filters, 1, strides=1, padding="same")(g)
    
    # Upsample g if spatial dims don't match x
    phi_g = layers.UpSampling2D(size=(theta_x.shape[1]//phi_g.shape[1], 
                                        theta_x.shape[2]//phi_g.shape[2]))(phi_g)
    
    add = layers.Add()([theta_x, phi_g])
    relu = layers.Activation("relu")(add)
    psi = layers.Conv2D(1, 1, strides=1, padding="same", activation="sigmoid")(relu)
    
    return layers.Multiply()([x, psi])  # attention-weighted skip connection

In [ ]:
# UNET Architecture
def convolution_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu", kernel_initializer="he_normal")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu", kernel_initializer="he_normal")(x)
    return x

# def building_unet(input_size=(256,256,3)):
#     inputs = layers.Input(input_size)

#     # Contracting path
#     c1 = convolution_block(inputs, 64)
#     p1 = layers.MaxPooling2D(pool_size=(2,2))(c1)

#     c2 = convolution_block(p1, 128)
#     p2 = layers.MaxPooling2D(pool_size=(2,2))(c2)

#     c3 = convolution_block(p2, 256)
#     p3 = layers.MaxPooling2D(pool_size=(2,2))(c3)

#     c4 = convolution_block(p3, 512)
#     p4 = layers.MaxPooling2D(pool_size=(2,2))(c4)

#     # Bottleneck
#     bn = convolution_block(p4, 1024)

#     # Expansive Path
#     u4 = layers.Conv2DTranspose(512, 2, strides=2, padding="same")(bn)
#     u4 = layers.Concatenate()([u4,c4])
#     c5 = convolution_block(u4, 512)

#     u3 = layers.Conv2DTranspose(256, 2, strides=2, padding="same")(c5)
#     u3 = layers.Concatenate()([u3,c3])
#     c6 = convolution_block(u3, 256)

#     u2 = layers.Conv2DTranspose(128, 2, strides=2, padding="same")(c6)
#     u2 = layers.Concatenate()([u2,c2])
#     c7 = convolution_block(u2, 128)

#     u1 = layers.Conv2DTranspose(64, 2, strides=2, padding="same")(c7)
#     u1 = layers.Concatenate()([u1,c1])
#     c8 = convolution_block(u1, 64)

#     outputs = layers.Conv2D(1, 1, activation="sigmoid")(c8)

#     model = Model(inputs, outputs, name="U-net")
#     return model

# model = building_unet()
# model.summary()

# Attention UNET Architecture
def attention_unet(input_size=(256,256,3)):
    inputs = layers.Input(input_size)

    # Contracting path
    c1 = convolution_block(inputs, 64)
    p1 = layers.MaxPooling2D(pool_size=(2,2))(c1)

    c2 = convolution_block(p1, 128)
    p2 = layers.MaxPooling2D(pool_size=(2,2))(c2)

    c3 = convolution_block(p2, 256)
    p3 = layers.MaxPooling2D(pool_size=(2,2))(c3)

    c4 = convolution_block(p3, 512)
    p4 = layers.MaxPooling2D(pool_size=(2,2))(c4)

    # Bottleneck
    bn = convolution_block(p4, 1024)

    # Expansive Path
    u4 = layers.Conv2DTranspose(512, 2, strides=2, padding="same")(bn)
    a4 = attention_gates(c4,bn,512)
    u4 = layers.Concatenate()([u4,a4])
    c5 = convolution_block(u4, 512)

    u3 = layers.Conv2DTranspose(256, 2, strides=2, padding="same")(c5)
    a3 = attention_gates(c3,c5,256)
    u3 = layers.Concatenate()([u3,a3])
    c6 = convolution_block(u3, 256)

    u2 = layers.Conv2DTranspose(128, 2, strides=2, padding="same")(c6)
    a2 = attention_gates(c2,c6,128)
    u2 = layers.Concatenate()([u2,a2])
    c7 = convolution_block(u2, 128)

    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding="same")(c7)
    a1 = attention_gates(c1,c7,64)
    u1 = layers.Concatenate()([u1,a1])
    c8 = convolution_block(u1, 64)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c8)

    model = Model(inputs, outputs, name="U-net")
    return model

model = attention_unet()
model.summary()



In [ ]:
# # Tversky loss function
# def tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7, smooth=1e-6):
#     y_true_f = tensorflow.keras.backend.flatten(y_true)
#     y_pred_f = tensorflow.keras.backend.flatten(y_pred)
#     tp = tensorflow.keras.backend.sum(y_true_f * y_pred_f)
#     fn = tensorflow.keras.backend.sum(y_true_f * (1 - y_pred_f))
#     fp = tensorflow.keras.backend.sum((1 - y_true_f) * y_pred_f)
#     tversky = (tp + smooth) / (tp + alpha*fp + beta*fn + smooth)
#     return 1 - tversky

In [ ]:
# compiling the model
model.compile(
    tensorflow.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# printing the progress
checkpoint_callback = tensorflow.keras.callbacks.ModelCheckpoint(
    'best_unet_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [ ]:
early_stop = tensorflow.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tensorflow.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=2,
    epochs=30,
    callbacks=[checkpoint_callback, early_stop, reduce_lr]
)

In [ ]:
# Predict it on the test set
preds = model.predict(X_val)
preds_binary = (preds > 0.5).astype('float32')

# Dice coefficient
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = (y_true_f * y_pred_f).sum()
    return (2. * intersection + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    intersection = (y_true * y_pred).sum()
    union = y_true.sum() + y_pred.sum() - intersection
    return (intersection + smooth) / (union + smooth)

dice_scores = [dice_coef(y_val[i], preds_binary[i]) for i in range(len(y_val))] 
iou_scores = [iou_score(y_val[i], preds_binary[i]) for i in range(len(y_val))]
print(f"Mean Dice: {np.mean(dice_scores):.4f}, Min: {np.min(dice_scores):.4f}, Std: {np.std(dice_scores):.4f}")
print(f"Mean IoU: {np.mean(iou_scores):.4f}, Min: {np.min(iou_scores):.4f}, Std: {np.std(iou_scores):.4f}")

In [ ]:
# Training and validation loss visualization
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label="Train loss")
plt.plot(history.history['val_loss'], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation loss")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Train accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Training accuracy')
plt.ylabel('validation accuracy')
plt.title('Training vs Validation accuracy Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(3):
    axes[i,0].imshow(X_val[i]); axes[i,0].set_title("Input")
    axes[i,1].imshow(y_val[i], cmap="gray"); axes[i,1].set_title("Ground Truth")
    axes[i,2].imshow(preds_binary[i], cmap="gray"); axes[i,2].set_title("Predicted")
plt.savefig("prediction_samples.png")

In [ ]:
# worst case 
worst_idx = np.argsort(dice_scores)[:5]  # 5 worst cases
for i in worst_idx:
    print(f"Index {i}, Dice: {dice_scores[i]:.4f}, IoU: {iou_scores[i]:.4f}")

fig, axes = plt.subplots(len(worst_idx), 3, figsize=(12, 4*len(worst_idx)))
for row, i in enumerate(worst_idx):
    axes[row,0].imshow(X_val[i]); axes[row,0].set_title(f"Input (idx {i})")
    axes[row,1].imshow(y_val[i].squeeze(), cmap="gray"); axes[row,1].set_title("Ground Truth")
    axes[row,2].imshow(preds_binary[i].squeeze(), cmap="gray"); axes[row,2].set_title("Predicted")
plt.tight_layout()
plt.savefig("worst_cases.png")
plt.show()